In [3]:
# 1) 빅카인즈 표본 xlsx 업로드 (버튼 → 파일 선택). 원문은 공개 링크로 받지 않는다
import ipywidgets as w

up = w.FileUpload(accept=".xlsx", multiple=False)
display(up)

FileUpload(value={}, accept='.xlsx', description='Upload')

In [6]:
# 2) 읽고 상위 20건만
import io
import pandas as pd

v = up.value
f = v[0] if isinstance(v, tuple) else next(iter(v.values()))  # ipywidgets 버전 차이
df = pd.read_excel(io.BytesIO(bytes(f["content"])))
df = df[df["분석제외 여부"].isna()].head(20).copy()

StopIteration: 

In [5]:
# 3) 모델 불러오기 (처음 1~2분)
import torch
from transformers import pipeline

clf = pipeline("text-classification", model="snunlp/KR-FinBert-SC",
               top_k=None, truncation=True,
               device=0 if torch.cuda.is_available() else -1)
print(clf.model.config.id2label)                        # 라벨 이름
print(getattr(clf.model.config, "_commit_hash", None))  # 모델 버전 기록

KeyboardInterrupt: 

In [ ]:
# 4) 입력 3가지: 제목 / 제목+본문 200자 / 제목+첫 문장
title = df["제목"].str.replace(r"\[[^\]]*\]", "", regex=True).str.strip()
body = (df["본문"].str.replace(r"^<[^>]*>", "", regex=True)
                  .str.replace("\n", " ").str.rstrip("."))
lead = body.str.split("다.", regex=False).str[0] + "다."

inputs = {"제목": title, "제목+본문": title + " " + body, "제목+첫문장": title + " " + lead}

In [ ]:
# 5) 점수: s = P(긍정) - P(부정). 라벨 이름이 다르면 아래 두 이름을 바꾼다
import time

def score(texts):
    out = clf(list(texts), batch_size=16)
    p = pd.DataFrame([{d["label"]: d["score"] for d in o} for o in out])
    return (p["positive"] - p["negative"]).values, p.idxmax(axis=1).values

t0 = time.time()
for k, texts in inputs.items():
    df[f"s_{k}"], df[f"lab_{k}"] = score(texts)
print(f"1건당 {(time.time() - t0) / (len(df) * len(inputs)):.3f}초")

In [ ]:
# 6) 결과 보기 (첫 확인용, 보고서 '결과'로 쓰지 않음)
cols = ["제목"] + [c for c in df.columns if c.startswith(("s_", "lab_"))]
display(df[cols].round(2))
for k in ["제목+본문", "제목+첫문장"]:
    print(k, "→ 제목만 넣었을 때와 라벨 일치율:", (df["lab_제목"] == df[f"lab_{k}"]).mean())